# makemore — an MLP character-level language model

Cleaned-up notes from Karpathy's *Zero to Hero* (makemore, the MLP video), following the Bengio et al. 2003 design.

The bigram model only looked one character back. Here we take a **fixed window of the previous few characters**, embed each into a small learned vector, feed the concatenation through a hidden layer, and predict the next character. Same softmax + cross-entropy objective as before — just a real (if small) neural net in the middle.

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

## The data

In [ ]:

# read in all the words
words = open('../data/names.txt', 'r').read().splitlines()
print(words[:8])
print(len(words))

## Vocabulary

`stoi` / `itos` map characters ↔ integers, with `.` = index 0 as the start/end token.

In [ ]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

## Building the dataset — context windows

`block_size` is how many previous characters we condition on (here 3). We slide a fixed window across each word: the window is the input `X`, the next character is the label `Y`. The window starts as all-`.` (padding) and rolls forward one character at a time (`context[1:] + [ix]`).

In [ ]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words:
  
  #print(w)
  context = [0] * block_size
  for ch in w + '.':
    ix = stoi[ch]
    X.append(context)
    Y.append(ix)
    # print(''.join(itos[i] for i in context), '--->', itos[ix])
    context = context[1:] + [ix] # crop and append
  
X = torch.tensor(X)
Y = torch.tensor(Y)

In [ ]:
print(X[:10])
print(Y[:10])

In [ ]:
X.shape, X.dtype, Y.shape, Y.dtype

## Train / dev / test split

Shuffle the words and cut 80% / 10% / 10%:
- **train** — fit the parameters,
- **dev (validation)** — tune hyperparameters (hidden size, learning rate, embedding dim),
- **test** — touch once at the very end, to get an honest final number.

Keeping dev and test separate is what stops you from overfitting your *choices* to the data.

In [ ]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?

def build_dataset(words):  
  X, Y = [], []
  for w in words:

    #print(w)
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
    #   print(''.join(itos[i] for i in context), '--->', itos[ix])
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

## Character embeddings — the lookup table `C`

Each of the 27 characters gets a small learned vector (here 2-D, later 10-D), stored as rows of `C`. Indexing `C[X]` looks up the embedding for every character in every context at once.

Indexing with a tensor of indices is the efficient equivalent of one-hot-encoding and matrix-multiplying: `F.one_hot(5) @ C` picks out exactly `C[5]`, but the indexing skips building the one-hot.

In [ ]:
C = torch.randn((27, 2))

In [ ]:
print(C[5])
print(C[torch.tensor([5,6,7])])
print(C[torch.tensor([5,6,7,7,4,5,7])])# duplicates

In [ ]:
F.one_hot(torch.tensor(5), num_classes=27).float() @ C

In [ ]:
print(C[X].shape)
print(X[13,2])
print(C[X][13,2])
print(C[1])

In [ ]:
emb = C[X]
emb.shape

## The hidden layer

`W1` maps the flattened context (block_size × embedding_dim = 3 × 2 = 6 inputs) to `hyper_h` hidden neurons, followed by a `tanh`. The number of hidden neurons is a hyperparameter.

In [ ]:
hyper_h = 100 # hypertunes parameter for number of neurons to use 
W1 = torch.randn((6, hyper_h)) 
b1 = torch.randn(hyper_h)

## Flattening the context embeddings

`emb` has shape `(N, block_size, emb_dim)` — we need `(N, block_size*emb_dim)` to feed the linear layer. Three ways, in increasing quality:
1. `torch.cat([emb[:,0,:], emb[:,1,:], emb[:,2,:]], 1)` — explicit, but hard-codes block_size.
2. `torch.cat(torch.unbind(emb, 1), 1)` — generic over block_size.
3. `emb.view(-1, 6)` — most efficient (a free reshape, no copy); `-1` lets torch infer the batch dimension.

In [ ]:
# we want to flatten now as we have 3 context window input vectors and each is of dimention 2 we want a single vector of 6 
print(torch.cat([emb[:,0,:], emb[:,1,:], emb[:,2,:]], 1)[0]) # this is one way of doing and pringint the first input array we would have
print(torch.cat([emb[:,0,:], emb[:,1,:], emb[:,2,:]], 1).shape) # this is one way of doing and pringint the first input array we would have

print(torch.cat(torch.unbind(emb,1),1)[0]) # this is another way which is better as we dont have to worry if i change my context size right now it was 3

print(emb.view(-1,6)[0]) # most efficeint way but we need to change this code if our embedding size changes. we done need to worry about number of samplesa as -1 tells
# torch to infer the shape.


In [ ]:
print((emb.view(-1,6) @ W1 + b1)[0])
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)

## Output layer and the loss (spelled out)

Second linear layer → logits → softmax → average negative log-likelihood, written by hand here so the pieces are visible.

In [ ]:
W2 = torch.randn((100, 27))
b2 = torch.randn(27)
logits = h @ W2 + b2
counts = logits.exp()
prob = counts / counts.sum(1, keepdims=True)
loss = -prob[torch.arange(228146), Y].log().mean()
print(loss)


## Scaling up: bigger embeddings, `cross_entropy`, and training

Now 10-D embeddings and a proper parameter list with `requires_grad`. The manual `exp / normalize / log` is replaced by `F.cross_entropy(logits, Y)`, which is the same loss but **numerically stable** (it subtracts the max logit before exponentiating, so nothing overflows) and faster. A forward pass alone is prediction; adding `loss.backward()` and a parameter update is training.

In [ ]:
X.shape, Y.shape

In [ ]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 10), generator=g)
W1 = torch.randn((30, 100), generator=g) # 200 is the hyper paramenet we use now. 
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]
for p in parameters:
  p.requires_grad = True
print('number of parameters', sum(p.nelement() for p in parameters))

In [ ]:
emb = C[X] # (32, 3, 2)
h = torch.tanh(emb.view(-1, 30) @ W1 + b1) # (32, 100)
logits = h @ W2 + b2 # (32, 27)
loss = F.cross_entropy(logits, Y)
print(loss)
# counts = logits.exp()
# prob = counts / counts.sum(1, keepdims=True)
# loss = -prob[torch.arange(228146), Y].log().mean()
# print(loss)

In [ ]:
# training untill now it was just forward pass now we add backward pass as well. this isslow as its on all

for i in range(10):
    emb = C[X] # (32, 3, 2)
    h = torch.tanh(emb.view(-1, 30) @ W1 + b1) # (32, 100)
    logits = h @ W2 + b2 # (32, 27)
    loss = F.cross_entropy(logits, Y)
    # print(loss.item())
    for p in parameters:
        p.grad = None

    loss.backward()

    for p in parameters:
        p.data += -0.1 * p.grad

print(loss.item())

## Finding a good learning rate

Too-high a learning rate diverges; too-low wastes time. A cheap way to find a sane range: sweep the rate exponentially from 1e-3 to 1 across steps and watch where the loss drops fastest.

In [ ]:
# we need to try learning rate if we are too high we will explode and not be stable abut if too small 
# then we ware not learing fast enough and our training time will need to be increase.
#  so here we are making a list of possible elarning rates from 0-1

lre = torch.linspace(-3, 0, 1000) # we are doing explonential 10^-3 is 0.001 and 0 is 1
lrs = 10**lre
# print(lrs)

## Minibatch training + learning-rate decay

Re-initialize a bigger net and train on random **minibatches** (32 examples via `torch.randint`) instead of the whole dataset each step — far more updates per second. The learning rate is stepped down partway through (**decay**): big steps early, small steps to settle.

In [ ]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 10), generator=g)
W1 = torch.randn((30, 300), generator=g) # 200 is the hyper paramenet we use now. 
b1 = torch.randn(300, generator=g)
W2 = torch.randn((300, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]
for p in parameters:
  p.requires_grad = True

lri = []
lossi = []
stepi = []


In [ ]:
# now we are doing batching this should be faster as we dont use the whole dataset.
for i in range(10000):
  
  # minibatch construct
  ix = torch.randint(0, Xtr.shape[0], (32,))
  
  # forward pass
  emb = C[Xtr[ix]] # (32, 3, 10)
  h = torch.tanh(emb.view(-1, 30) @ W1 + b1) # (32, 200)
  logits = h @ W2 + b2 # (32, 27)
  loss = F.cross_entropy(logits, Ytr[ix])
  #print(loss.item())
  
  # backward pass
  for p in parameters:
    p.grad = None
  loss.backward()
  
  # update
  # lr = lrs[(int)(i/1000)]
  lr = 0.1 if i < 100000 else 0.01
  for p in parameters:
    p.data += -lr * p.grad

  # track stats
  lri.append(lr)
  stepi.append(i)
  lossi.append(loss.log10().item())

print(loss.item())

## Evaluating: train vs dev loss

Compare loss on train and dev. Close together → likely underfitting (make the model bigger); a large gap → overfitting.

In [ ]:
# this is the whole dataset loss 
emb = C[X] # (32, 3, 2) 
h = torch.tanh(emb.view(-1, 30) @ W1 + b1) # (32, 100)
logits = h @ W2 + b2 # (32, 27)
loss = F.cross_entropy(logits, Y)
print(loss)

In [ ]:
plt.plot(lri, lossi)

In [ ]:
emb = C[Xtr] # (32, 3, 2)
h = torch.tanh(emb.view(-1, 30) @ W1 + b1) # (32, 100)
logits = h @ W2 + b2 # (32, 27)
loss = F.cross_entropy(logits, Ytr)
loss

In [ ]:

emb = C[Xdev] # (32, 3, 2)
h = torch.tanh(emb.view(-1, 30) @ W1 + b1) # (32, 100)
logits = h @ W2 + b2 # (32, 27)
loss = F.cross_entropy(logits, Ydev)
loss

In [ ]:

plt.plot(stepi, lossi)

## Visualizing the learned embeddings

With a 2-D embedding you can plot the characters directly — vowels and similar letters tend to cluster.

In [ ]:
plt.figure(figsize=(8,8))
plt.scatter(C[:,0].data, C[:,1].data, s=200)
for i in range(C.shape[0]):
    plt.text(C[i,0].item(), C[i,1].item(), itos[i], ha="center", va="center", color='white')
plt.grid('minor')

## Sampling from the model

Start from all-`.` context, repeatedly sample the next character from the softmax and roll the window forward, until `.` is drawn.

In [ ]:

# sample from the model
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):
    
    out = []
    context = [0] * block_size # initialize with all ...
    while True:
      emb = C[torch.tensor([context])] # (1,block_size,d)
      h = torch.tanh(emb.view(1, -1) @ W1 + b1)
      logits = h @ W2 + b2
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break
    
    print(''.join(itos[i] for i in out))